EEG-to-Video Inference Script.

This script orchestrates the entire process of generating a video from EEG
signals using a pre-trained multi-stage model pipeline. It performs the
final denoising and decoding stage of the EEG2Video framework.

The process involves:
1.  Loading a fine-tuned 3D U-Net model for video diffusion.
2.  Loading pre-generated EEG semantic embeddings to guide the generation process.
3.  Loading an initial latent video sequence, which is a "draft" generated by
    a Seq2Seq model and optionally noised by the DANA module.
4.  Using a custom diffusion pipeline to perform iterative denoising, conditioned
    on the EEG embeddings, to generate the final video.
5.  Saving the generated video as a GIF file.

The script supports different inference modes for ablation studies:
- 'full': The complete pipeline using both Seq2Seq and DANA.
- 'no_dana': Bypasses the DANA module, starting denoising from the clean
             Seq2Seq latents.
- 'no_seq2seq': Bypasses both Seq2Seq and DANA, starting from pure Gaussian noise.

## CONFIGURATION

In [1]:
import torch

CONFIG = {
    "seed": 42,

    "pretrained_model_id": "CompVis/stable-diffusion-v1-4",
    "finetuned_unet_path": "checkpoints/video_diffusion_finetuned",

    "eeg_embeddings_path": "data/metadata/semantic_embeddings.pt",
    "latents_seq2seq_path": "data/metadata/seq2seq_prediction.pt",
    "latents_dana_path": "data/metadata/noise_videos_latents.pt",

    "output_dir": "outputs/generated_videos",

    "video_length": 12,
    "height": 288,
    "width": 512,
    "num_inference_steps": 100,
    "guidance_scale": 12.5,

    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "dtype": torch.float16,
}

## Set seed

In [2]:
import random
import os
import numpy as np

def set_seed(seed: int):
    """Sets the random seed for reproducibility.

    Args:
        seed (int): The seed to use for all random number generators.
    """
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Seed set to {seed}")

set_seed(CONFIG["seed"])

Seed set to 42


## Load Data

In [ ]:
from einops import rearrange

def load_data() -> tuple:
    """Loads all necessary data for inference.

    Args:
        config (InferenceConfig): The inference configuration object.

    Returns:
        tuple: A tuple containing (eeg_embeddings, latents_seq2seq, latents_dana).
    """
    print("📂 Loading data...")
    eeg_embeddings = torch.load(CONFIG["eeg_embeddings_path"], map_location='cpu')
    print(f"  - Loaded EEG embeddings: {eeg_embeddings.shape}")

    latents_seq2seq = torch.from_numpy(np.load(CONFIG["latents_seq2seq_path"])).to(CONFIG["dtype"])
    latents_seq2seq = rearrange(latents_seq2seq, 'b f c h w -> b c f h w')
    print(f"  - Loaded Seq2Seq latents: {latents_seq2seq.shape}")

    latents_dana = torch.load(CONFIG["latents_dana_path"], map_location='cpu').to(CONFIG["dtype"])
    latents_dana = rearrange(latents_dana, 'b f c h w -> b c f h w')
    print(f"  - Loaded DANA noised latents: {latents_dana.shape}")
    
    return eeg_embeddings, latents_seq2seq, latents_dana

## Load pipeline

In [ ]:
from diffusers import AutoencoderKL, DDIMScheduler

from models.unet_fixed import UNet3DConditionModel
from models.pipeline_tuneeeg2video import TuneAVideoPipeline as EEG2VideoPipeline
from utils.util import save_videos_grid


def load_pipeline() -> EEG2VideoPipeline:
    """Loads and configures the video generation pipeline.

    Args:
        config (InferenceConfig): The inference configuration object.

    Returns:
        EEG2VideoPipeline: The configured video generation pipeline.
    """
    print(f"Loading U-Net from fine-tuned checkpoint: {CONFIG["finetuned_unet_path"]}")
    unet = UNet3DConditionModel.from_pretrained(
        CONFIG["finetuned_unet_path"], subfolder='unet', torch_dtype=CONFIG["dtype"]
    ).to(CONFIG["device"])

    print(f"Loading VAE and Tokenizer from pre-trained model: {CONFIG["pretrained_model_id"]}")
    vae = AutoencoderKL.from_pretrained(
        CONFIG["pretrained_model_id"], subfolder="vae", torch_dtype=CONFIG["dtype"]
    ).to(CONFIG["device"])
    
    tokenizer = None  # Tokenizer is not directly used in this EEG-conditioned pipeline

    scheduler = DDIMScheduler.from_pretrained(
        CONFIG["pretrained_model_id"], subfolder="scheduler"
    )

    pipeline = EEG2VideoPipeline(
        vae=vae,
        tokenizer=tokenizer,
        unet=unet,
        scheduler=scheduler,
    ).to(CONFIG["device"])

    # Apply memory-saving optimizations
    pipeline.enable_xformers_memory_efficient_attention()
    pipeline.enable_vae_slicing()
    
    print("✅ Pipeline loaded and configured successfully.")
    return pipeline

## Run inference

In [ ]:
from tqdm import tqdm
from pathlib import Path

def run_inference(
    pipeline: EEG2VideoPipeline,
    eeg_embeddings: torch.Tensor,
    latents_seq2seq: torch.Tensor,
    latents_dana: torch.Tensor,
    mode: str,
):
    """Runs the main inference loop to generate and save videos.

    Args:
        pipeline (EEG2VideoPipeline): The configured generation pipeline.
        eeg_embeddings (torch.Tensor): Tensor of semantic EEG embeddings.
        latents_seq2seq (torch.Tensor): Clean latents from the Seq2Seq model.
        latents_dana (torch.Tensor): Noised latents from the DANA module.
        mode (str): The inference mode ('full', 'no_dana', or 'no_seq2seq').
        config (InferenceConfig): The inference configuration object.
    """
    output_path = Path(CONFIG["output_dir"]) / mode
    output_path.mkdir(parents=True, exist_ok=True)
    print(f"🚀 Starting inference in '{mode}' mode. Videos will be saved to: {output_path}")

    # Create negative conditioning from the average EEG embedding
    negative_eeg = eeg_embeddings.mean(dim=0, keepdim=True)
    
    num_samples = len(eeg_embeddings)
    for i in tqdm(range(num_samples), desc=f"Generating videos ({mode})"):
        # Select the initial latent tensor based on the inference mode
        if mode == 'full':
            initial_latents = latents_dana[i:i + 1].to(CONFIG["device"])
        elif mode == 'no_dana':
            initial_latents = latents_seq2seq[i:i + 1].to(CONFIG["device"])
        elif mode == 'no_seq2seq':
            # Start from pure Gaussian noise, bypassing Seq2Seq and DANA
            initial_latents = None
        else:
            raise ValueError(f"Invalid mode: {mode}")

        # Generate video using the pipeline
        video = pipeline(
            eeg=eeg_embeddings[i:i + 1],
            negative_eeg=negative_eeg,
            latents=initial_latents,
            video_length=CONFIG["video_length"],
            height=CONFIG["height"],
            width=CONFIG["width"],
            num_inference_steps=CONFIG["num_inference_steps"],
            guidance_scale=CONFIG["guidance_scale"],
        ).videos

        # Save the generated video
        save_videos_grid(video, output_path / f"{i:04d}.gif")

## Define main function

In [ ]:
import argparse

def main():
    """Main function to parse arguments and run the inference pipeline."""
    parser = argparse.ArgumentParser(description="Run EEG-to-Video inference.")
    parser.add_argument(
        "--mode",
        type=str,
        default="full",
        choices=["full", "no_dana", "no_seq2seq"],
        help="Inference mode for ablation study: "
             "'full' uses the complete pipeline, "
             "'no_dana' uses clean Seq2Seq latents, "
             "'no_seq2seq' starts from pure noise."
    )
    args = parser.parse_args()
    
    
    try:
        pipeline = load_pipeline()
        eeg_embeddings, latents_seq2seq, latents_dana = load_data()
        
        run_inference(
            pipeline=pipeline,
            eeg_embeddings=eeg_embeddings,
            latents_seq2seq=latents_seq2seq,
            latents_dana=latents_dana,
            mode=args.mode,
        )
        
        print("\n🎉 Inference complete!")
        
    except Exception as e:
        print(f"\n💥 An error occurred during inference: {e}")
        import traceback
        traceback.print_exc()

## Start

In [ ]:
if __name__ == "__main__":
    main()